# Support Ticket Classification - Training Notebook

Trains the **category** classifier and the **priority** classifier used by the API.

Pipeline: raw text -> clean/tokenize (spaCy) -> TF-IDF -> Calibrated LinearSVC

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
import pandas as pd
from app.utils.text_cleaning import preprocess

df = pd.read_csv('../data/tickets.csv')
df.head()

## 1. Exploratory look at the data

In [ ]:
print(df['category'].value_counts())
print()
print(df['priority'].value_counts())

## 2. Text cleaning & tokenization

In [ ]:
df['clean_text'] = df['text'].apply(preprocess)
df[['text', 'clean_text']].head(10)

## 3. Train the category classifier

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['category'], test_size=0.2, random_state=42, stratify=df['category']
)

category_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2))),
    ('clf', CalibratedClassifierCV(LinearSVC(class_weight='balanced'), cv=3)),
])
category_pipeline.fit(X_train, y_train)
print(classification_report(y_test, category_pipeline.predict(X_test), zero_division=0))

## 4. Train the priority classifier

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['priority'], test_size=0.2, random_state=42, stratify=df['priority']
)

priority_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2))),
    ('clf', CalibratedClassifierCV(LinearSVC(class_weight='balanced'), cv=3)),
])
priority_pipeline.fit(X_train, y_train)
print(classification_report(y_test, priority_pipeline.predict(X_test), zero_division=0))

## 5. Save models for the API to load

In [ ]:
import joblib, pathlib
out_dir = pathlib.Path('../saved_models')
out_dir.mkdir(exist_ok=True)
joblib.dump(category_pipeline, out_dir / 'category_model.joblib')
joblib.dump(priority_pipeline, out_dir / 'priority_model.joblib')
print('Saved both models to', out_dir.resolve())